# Novo benchmark externo Mirage-Test - Fase 3.5 congelada

Este notebook baixa uma selecao fixa de 1.000 reais e 1.000 sinteticas, confere hashes e faz inferencia somente quando a celula de avaliacao for executada manualmente. Nunca use estes exemplos para treino, calibracao ou escolha de hiperparametros. Os resultados historicos permanecem intactos.


In [1]:
# 1. GPU e dependências do runtime.
import torch, sys, subprocess
assert torch.cuda.is_available(), 'Conecte uma GPU do Colab antes de continuar'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cryptography', 'huggingface_hub', 'pyarrow', 'scikit-learn'])
print('Dependências disponíveis.', flush=True)

GPU: Tesla T4
Dependências disponíveis.


In [2]:
# 2. Drive e pastas novas; não altera experimentos antigos.
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
for name in ('credenciais', 'dados_externos', 'modelos', 'resultados'):
    (ROOT / name).mkdir(parents=True, exist_ok=True)
print('Drive:', ROOT, '| livre reportado:', round(shutil.disk_usage(ROOT).free / 1e9, 2), 'GB')
print('/content livre:', round(shutil.disk_usage('/content').free / 1e9, 2), 'GB')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive: /content/drive/MyDrive/TCC_Final | livre reportado: 7.49 GB
/content livre: 204.33 GB


In [3]:
# 3. Desbloqueio seguro da credencial GitHub existente (formatos antigo e novo).
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
credential_path = ROOT / 'credenciais/github.enc'
credential = json.loads(credential_path.read_text(encoding='utf-8'))
if not isinstance(credential, dict) or 'salt' not in credential or ('token' in credential) == ('ciphertext' in credential):
    raise ValueError('Formato da credencial GitHub não reconhecido')
if 'ciphertext' in credential:
    if credential.get('version') != 1: raise ValueError('Versão de credencial não suportada')
    iterations, encrypted = int(credential['iterations']), credential['ciphertext']
else:
    iterations, encrypted = 600000, credential['token']
if iterations < 600000: raise ValueError('KDF insuficiente')
password = getpass.getpass('Senha de desbloqueio do GitHub: ')
key = base64.urlsafe_b64encode(PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=base64.b64decode(credential['salt']), iterations=iterations).derive(password.encode()))
try:
    token = Fernet(key).decrypt(encrypted.encode()).decode()
except InvalidToken:
    raise ValueError('Senha incorreta ou credencial corrompida') from None
finally:
    del password, key
print('Credencial desbloqueada nesta sessão.')

Credencial desbloqueada nesta sessão.


In [4]:
# 4. Snapshot privado imutável da branch, somente leitura; sem token em URL ou Git.
import io, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
API = 'https://api.github.com/repos/vicemnovo/TCC_Final'
BRANCH = 'main'
def github_read(url):
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Mirage-Colab'})
    try:
        with urllib.request.urlopen(req, timeout=120) as response:
            chunks, size, notice = [], 0, time.monotonic()
            while block := response.read(1024 * 1024):
                chunks.append(block); size += len(block)
                if time.monotonic() - notice >= 30:
                    print(f'Download do código: {size / 1048576:.1f} MiB', flush=True); notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code}; confira token, repositório e branch') from None
try:
    commit = json.loads(github_read(f'{API}/branches/{BRANCH}'))['commit']['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', commit)
    CODE_ROOT = Path('/content/tcc_codigo') / commit
    required = ('experimentos/benchmark_mirage_2000.py', 'experimentos/avaliar_benchmark_2000.py', 'Notebooks/fase_1/fase1_vit.ipynb', 'Notebooks/fase_3/fase3_3/fase3.3_compressao.ipynb')
    if not CODE_ROOT.exists():
        print('Baixando snapshot do GitHub...', flush=True)
        archive = github_read(f'{API}/zipball/{commit}')
        CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='tcc35_', dir=CODE_ROOT.parent) as temporary:
            staged = Path(temporary)
            with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
                for item in bundle.infolist():
                    parts = Path(item.filename).parts
                    if item.is_dir() or len(parts) < 2: continue
                    relative = Path(*parts[1:])
                    if relative.is_absolute() or '..' in relative.parts: raise ValueError('Snapshot contém caminho inseguro')
                    target = staged / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with bundle.open(item) as source, target.open('wb') as output: shutil.copyfileobj(source, output)
            if not all((staged / name).is_file() for name in required):
                raise FileNotFoundError('Publique primeiro o benchmark Mirage na branch')
            staged.rename(CODE_ROOT)
    assert all((CODE_ROOT / name).is_file() for name in required)
finally:
    del token
print('Código sincronizado:', commit)

Baixando snapshot do GitHub...
Código sincronizado: 859253a8b9d0ffcdac903bfa7db846ce0d65a99d


In [5]:
# 5. Comandos e progresso periódico; nenhum treino é iniciado aqui.
import os, queue, subprocess, sys, threading, time
BENCH = CODE_ROOT / 'experimentos/benchmark_mirage_2000.py'
for path in (BENCH,):
    if not path.is_file(): raise FileNotFoundError(path)
ENV = os.environ.copy()
ENV['TCC_CODE_ROOT'] = str(CODE_ROOT)
def run_live(label, command):
    print(label + ': iniciando', flush=True)
    process = subprocess.Popen(command, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = queue.Queue()
    def reader():
        for line in process.stdout: lines.put(line)
        lines.put(None)
    threading.Thread(target=reader, daemon=True).start()
    begin = last = time.monotonic()
    while True:
        try:
            line = lines.get(timeout=5)
            if line is None: break
            print(line, end='', flush=True)
            last = time.monotonic()
        except queue.Empty:
            if time.monotonic() - last >= 30:
                print(f'{label}: ativo há {(time.monotonic()-begin)/60:.1f} min; sem nova saída', flush=True)
                last = time.monotonic()
    code = process.wait()
    if code: raise RuntimeError(f'{label} falhou (código {code}); veja o log acima')
    print(f'{label}: concluído em {(time.monotonic()-begin)/60:.1f} min', flush=True)
print('Código:', CODE_ROOT)
# Credencial Hugging Face opcional; nunca ler userdata.get() neste runtime.
# O dataset e publico, mas autenticacao de leitura pode evitar limites de taxa.
hf_path = ROOT / 'credenciais/huggingface.enc'
if hf_path.is_file():
    hf_credential = json.loads(hf_path.read_text(encoding='utf-8'))
    if hf_credential.get('version') != 1 or int(hf_credential.get('iterations', 0)) < 600000:
        raise ValueError('Formato da credencial Hugging Face nao reconhecido')
    hf_password = getpass.getpass('Senha da credencial Hugging Face: ')
    hf_key = base64.urlsafe_b64encode(PBKDF2HMAC(
        algorithm=hashes.SHA256(), length=32,
        salt=base64.b64decode(hf_credential['salt']),
        iterations=int(hf_credential['iterations'])).derive(hf_password.encode()))
    try:
        ENV['HF_TOKEN'] = Fernet(hf_key).decrypt(hf_credential['ciphertext'].encode()).decode()
    except InvalidToken:
        raise ValueError('Senha incorreta ou credencial Hugging Face corrompida') from None
    finally:
        del hf_password, hf_key
    print('Hugging Face: autenticacao de leitura disponivel nesta sessao.')
else:
    ENV.pop('HF_TOKEN', None)
    print('Hugging Face: acesso publico sem token; podem ocorrer limites de taxa.')


Código: /content/tcc_codigo/859253a8b9d0ffcdac903bfa7db846ce0d65a99d
Hugging Face: autenticacao de leitura disponivel nesta sessao.


## Passos manuais

Execute o piloto, depois prepare e copie o conjunto. Confira a configuracao congelada antes de executar a avaliacao unica. A primeira preparacao baixa arquivos individuais do dataset; a cada sessao seguinte, apenas a copia do Drive para `/content` e necessaria.


In [6]:
# 6. PILOTO: somente indice, contagens e commit; nao baixa imagens nem infere.
run_live('piloto Mirage', [sys.executable, '-u', str(BENCH), 'pilot'])


piloto Mirage: iniciando
Índice Mirage-Test: 49000 linhas | seleção fixa: 2000 | commit: 820a191bb0844ae74a1bd9eb57b28268257a4053
Domínios/classes: {('Animal', 0): 250, ('Animal', 1): 250, ('Human', 0): 250, ('Human', 1): 250, ('Object', 0): 250, ('Object', 1): 250, ('Scene', 0): 250, ('Scene', 1): 250}
Piloto apenas de metadados; não baixa as 2.000 imagens nem cria seleção no Drive. | índice SHA-256: 937a575063d0846c5b8eb11a8dfcc723c52e5b1c5ef98d12362802371d2f62b7 | primeira imagem: Animal/0_real/3e0a50ee4a34d5b25a45a07b23623fec01312b1c54b16435def7787fd294f811.jpg
piloto Mirage: concluído em 0.0 min


In [7]:
# 7. PREPARACAO MANUAL: fixa o plano antes do download; salva imagens e hashes no Drive.
# Se desconectar, execute novamente para retomar o mesmo plano.
run_live('preparacao Mirage', [sys.executable, '-u', str(BENCH), 'prepare'])


preparacao Mirage: iniciando
Índice Mirage-Test: 49000 linhas | seleção fixa: 2000 | commit: 820a191bb0844ae74a1bd9eb57b28268257a4053
Domínios/classes: {('Animal', 0): 250, ('Animal', 1): 250, ('Human', 0): 250, ('Human', 1): 250, ('Object', 0): 250, ('Object', 1): 250, ('Scene', 0): 250, ('Scene', 1): 250}
Plano fixado antes das imagens: /content/drive/MyDrive/TCC_Final/dados_externos/benchmark_externo_mirage_2000_v1.incompleto/plan.json
Sobreposição: OpenFake train 500/2032
Sobreposição: OpenFake train 1000/2032
Sobreposição: OpenFake train 1500/2032
Sobreposição: OpenFake train 2000/2032
Sobreposição: OpenFake test 500/1947
Sobreposição: OpenFake test 1000/1947
Sobreposição: OpenFake test 1500/1947
preparacao Mirage: ativo há 0.7 min; sem nova saída
Preparação Mirage-Test: 50/2000
preparacao Mirage: ativo há 1.3 min; sem nova saída
Preparação Mirage-Test: 100/2000
preparacao Mirage: ativo há 2.0 min; sem nova saída
Preparação Mirage-Test: 150/2000
preparacao Mirage: ativo há 2.7 min

In [8]:
# 8. COPIA LOCAL: repita em cada nova sessao do Colab.
run_live('copia local Mirage', [sys.executable, '-u', str(BENCH), 'stage'])


copia local Mirage: iniciando
Cópia/verificação: 100/2000
Cópia/verificação: 200/2000
Cópia/verificação: 300/2000
Cópia/verificação: 400/2000
Cópia/verificação: 500/2000
Cópia/verificação: 600/2000
Cópia/verificação: 700/2000
Cópia/verificação: 800/2000
Cópia/verificação: 900/2000
Cópia/verificação: 1000/2000
Cópia/verificação: 1100/2000
Cópia/verificação: 1200/2000
Cópia/verificação: 1300/2000
Cópia/verificação: 1400/2000
Cópia/verificação: 1500/2000
Cópia/verificação: 1600/2000
Cópia/verificação: 1700/2000
Cópia/verificação: 1800/2000
Cópia/verificação: 1900/2000
Cópia/verificação: 2000/2000
Benchmark local pronto: /content/benchmark_externo_mirage_2000_v1
copia local Mirage: concluído em 0.4 min


In [9]:
# 9. CONFERENCIA: leitura do manifesto e do checkpoint, sem inferencia.
from pathlib import Path
import json, hashlib
manifest = ROOT / 'dados_externos/benchmark_externo_mirage_2000_v1/manifest.json'
checkpoint = ROOT / 'modelos/fase3_5_cfsmall_v1/phase3_final.pt'
assert manifest.is_file() and checkpoint.is_file()
plan = json.loads(manifest.read_text(encoding='utf-8'))
assert len(plan['rows']) == 2000 and plan['exact_overlap_checked']
assert sum(r['label'] == 0 for r in plan['rows']) == 1000
assert sum(r['label'] == 1 for r in plan['rows']) == 1000
print('Manifesto congelado:', manifest)
print('Commit dataset:', plan['revision'])
print('Modelo congelado:', plan['model']['checkpoint_sha256'])
print('Temperatura:', plan['model']['temperature'], '| limiar:', plan['model']['threshold'])
print('Copia local:', (Path('/content/benchmark_externo_mirage_2000_v1/manifest.json')).is_file())

from collections import Counter
for label, title in ((0, 'reais'), (1, 'sinteticas')):
    subset = [row for row in plan['rows'] if row['label'] == label]
    formats = Counter(row['format'] for row in subset)
    groups = Counter((row['domain'], row['group']) for row in subset)
    short_sides = sorted(min(row['size']) for row in subset)
    print(title, '| formatos:', dict(formats), '| grupos:', len(groups),
          '| lado curto p10/med/p90:',
          (short_sides[len(subset)//10], short_sides[len(subset)//2],
           short_sides[9*len(subset)//10]))


Manifesto congelado: /content/drive/MyDrive/TCC_Final/dados_externos/benchmark_externo_mirage_2000_v1/manifest.json
Commit dataset: 820a191bb0844ae74a1bd9eb57b28268257a4053
Modelo congelado: 1dd78d4cd32106d1471d388fa3a96a094c3906e7860ed4ad5ec3d7acad9b9045
Temperatura: 0.4409470856189728 | limiar: 0.44499999999999995
Copia local: True
reais | formatos: {'JPEG': 1000} | grupos: 4 | lado curto p10/med/p90: (2418, 3648, 4480)
sinteticas | formatos: {'JPEG': 1000} | grupos: 24 | lado curto p10/med/p90: (1024, 1024, 1024)


In [10]:
# 10. AVALIACAO EXTERNA MANUAL E UNICA da Fase 3.5.
# Execute so apos conferir as 2.000 imagens e o modelo congelado.
# O script recusa resultado anterior existente e nunca altera o checkpoint.
run_live('avaliacao externa Mirage 3.5', [sys.executable, '-u', str(BENCH), 'evaluate'])


avaliacao externa Mirage 3.5: iniciando
Integridade antes da inferência: 100/2000
Integridade antes da inferência: 200/2000
Integridade antes da inferência: 300/2000
Integridade antes da inferência: 400/2000
Integridade antes da inferência: 500/2000
Integridade antes da inferência: 600/2000
Integridade antes da inferência: 700/2000
Integridade antes da inferência: 800/2000
Integridade antes da inferência: 900/2000
Integridade antes da inferência: 1000/2000
Integridade antes da inferência: 1100/2000
Integridade antes da inferência: 1200/2000
Integridade antes da inferência: 1300/2000
Integridade antes da inferência: 1400/2000
Integridade antes da inferência: 1500/2000
Integridade antes da inferência: 1600/2000
Integridade antes da inferência: 1700/2000
Integridade antes da inferência: 1800/2000
Integridade antes da inferência: 1900/2000
Integridade antes da inferência: 2000/2000
Fase 3.5 congelada | GPU=Tesla T4 | T=0.440947 | limiar=0.445000
Inferência externa: 100/2000
Inferência exte

In [11]:
# 11. RESULTADO SOMENTE-LEITURA.
result = ROOT / 'resultados/benchmark_externo_mirage_2000_v1/fase3_5_cfsmall_v1/metrics.json'
print(json.dumps(json.loads(result.read_text(encoding='utf-8')), indent=2, ensure_ascii=False) if result.is_file() else 'Avaliacao ainda nao executada.')


{
  "model": "fase3_5_cfsmall_v1",
  "n": 2000,
  "real": 1000,
  "fake": 1000,
  "accuracy": 0.6045,
  "accuracy_ci95_wilson": [
    0.5828901539119635,
    0.6257091832062547
  ],
  "balanced_accuracy": 0.6045,
  "auc": 0.679972,
  "precision": 0.7119675456389453,
  "recall_fake": 0.351,
  "specificity_real": 0.858,
  "f1": 0.47019423978566643,
  "tn": 858,
  "fp": 142,
  "fn": 649,
  "tp": 351,
  "temperature_frozen": 0.4409470856189728,
  "threshold_frozen": 0.44499999999999995,
  "upscaled_short_side_below_224": 0,
  "checkpoint_sha256": "1dd78d4cd32106d1471d388fa3a96a094c3906e7860ed4ad5ec3d7acad9b9045",
  "manifest_sha256": "4b3658f880e022957557933fd662c69485c19a5ed9a04e2e09c32ce485737190",
  "evaluation_script_sha256": "6b1f4ca329f1882fd401e6fdbbcada64cf74201a0ab55808b008c721e80b58a3",
  "architecture_source_sha256": "3db23c93de1130c4b55a1b1ad2f3f522005e5e2dd3d15733698f1f2418eb7c7b",
  "architecture_extraction_sha256": "ad5db5ee692cd67bc3838ff814b36a179f3eb1acc2cdd77cf5fee41a961